# Melanoma experiments

Settings > Accelerator > **GPU P100**, and Internet **On**.

Everything is resumable. If the session dies, start it again and it skips
whatever already finished.

In [ ]:
import os, subprocess, shutil, time

t0 = time.time()

# Kaggle already has torch and timm. albumentations is usually there too, but
# pin it if the version is old enough to lack Affine.
try:
    import albumentations as A
    A.Affine
    print('albumentations', A.__version__, 'OK')
except Exception:
    subprocess.run(['pip', 'install', '-q', 'albumentations>=2.0'], check=False)

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('No GPU. Settings > Accelerator > GPU, then rerun.')

In [ ]:
# --- find the input dataset -------------------------------------------
INPUT = None
for d in sorted(os.listdir('/kaggle/input')):
    if os.path.exists(f'/kaggle/input/{d}/folds.csv'):
        INPUT = f'/kaggle/input/{d}'
        break
print('input:', INPUT)
assert INPUT, 'Add the melanoma dataset under Data on the right.'
print(os.listdir(INPUT))

In [ ]:
# --- unpack the photos to local SSD (much faster to read than /kaggle/input)
WORK = '/kaggle/working'
DATA = f'{WORK}/data'
os.makedirs(DATA, exist_ok=True)

if not os.path.exists(f'{DATA}/train_512'):
    tar = [f for f in os.listdir(INPUT) if f.endswith('.tar')]
    if tar:
        print('untarring', tar[0], '...')
        subprocess.run(['tar', '-xf', f'{INPUT}/{tar[0]}', '-C', DATA], check=True)
        # the tar contains a top folder, flatten it
        inner = [d for d in os.listdir(DATA) if os.path.isdir(f'{DATA}/{d}')]
        if len(inner) == 1 and not os.path.exists(f'{DATA}/train_512'):
            for item in os.listdir(f'{DATA}/{inner[0]}'):
                shutil.move(f'{DATA}/{inner[0]}/{item}', f'{DATA}/{item}')
    else:
        DATA = INPUT

print('data dir:', DATA)
print(sorted(os.listdir(DATA))[:8])
print('train photos:', len(os.listdir(f'{DATA}/train_512')))

In [ ]:
# --- put our source files where python can import them ------------------
SRC = f'{WORK}/src'
os.makedirs(SRC, exist_ok=True)
for d in os.listdir('/kaggle/input'):
    cand = f'/kaggle/input/{d}/src'
    if os.path.isdir(cand):
        for f in os.listdir(cand):
            if f.endswith('.py'):
                shutil.copy(f'{cand}/{f}', f'{SRC}/{f}')
print(sorted(os.listdir(SRC)))

In [ ]:
# --- run the programme --------------------------------------------------
# time_budget_h leaves headroom inside Kaggle's 9 hour session limit so the
# run stops cleanly and saves, instead of being killed mid-fold.
BUDGET = 7.5 - (time.time() - t0) / 3600

!cd {SRC} && python experiment_runner.py \
    --data_dir {DATA} \
    --out_dir {WORK}/reports \
    --batch_size 64 \
    --num_workers 2 \
    --time_budget_h {BUDGET:.2f} \
    --save_weights

In [ ]:
# --- build the report ---------------------------------------------------
!cd {SRC} && python report_results.py --in_dir {WORK}/reports

import pandas as pd
pd.read_csv(f'{WORK}/reports/results.csv')

In [ ]:
# --- keep the outputs, drop the unpacked photos so the output stays small
shutil.rmtree(f'{WORK}/data', ignore_errors=True)
print('outputs kept:')
for root, _, files in os.walk(f'{WORK}/reports'):
    for f in files:
        p = os.path.join(root, f)
        print(f'  {os.path.getsize(p)/1e6:8.2f} MB  {p}')